# Chapter `2.4` - `PROJECT` - Wedding Planner

## Setup and configuration

#### Importing the necessary libraries

In [47]:
# Base utils.
from os import getenv
from dotenv import load_dotenv
import warnings

from typing import Dict, Any
from IPython.display import Markdown

# Model init. and invocation
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent
from langchain.tools import tool

# MCP server
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.utilities import SQLDatabase
from tavily import TavilyClient

# State management
from langchain.agents import AgentState
from langchain.tools import ToolRuntime
from langgraph.types import Command

# Probing
from sqlalchemy import create_engine, text

#### Environent settings

In [7]:
load_dotenv()

try:
    OLLAMA_MODEL = getenv("OLLAMA_MODEL", "")
    if not len(OLLAMA_MODEL):
        raise EnvironmentError("Missing Ollama model configuration in environment.")
except EnvironmentError as ee:
    print(f"ERROR: {ee}")

#### Supressing warnings

In [8]:
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Specific LangChain / LangGraph noise
warnings.filterwarnings("ignore", module="langchain")
warnings.filterwarnings("ignore", module="langgraph")

## Tools setup


### 1. Travel MCP
- Using the `kiwi` MCP server for travel planning.

In [ ]:
kiwi_travel_client = MultiServerMCPClient(
    {"travel_server": {"transport": "streamable_http", "url": "https://mcp.kiwi.com"}}
)

tools = await kiwi_travel_client.get_tools()

### 2. Web Search MCP
- Using the **Tavily Search** API for fetching latest information from the web.

In [17]:
tavily_client = TavilyClient()


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""

    return tavily_client.search(query)

### 3. Database MCP
- The **SQLite** database MCP for fetching the playlist information from a local database.

In [12]:
db_local_path = r"resources/Chinook.db"
db = SQLDatabase.from_uri(rf"sqlite:///{db_local_path}")


@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""
    try:
        return db.run(query)  # type: ignore
    except Exception as e:
        return f"Error querying database: {e}"

## Agent state management

In [14]:
class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Agent architecture


### Subagents

#### 1. Travel planner

In [24]:
TRAVEL_PLANNER_SYS_PROMPT = """
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    Once you have found the best options, let the user know your shortlist of options.
    """

travel_agent = create_agent(
    model=OLLAMA_MODEL,
    tools=tools,
    system_prompt=TRAVEL_PLANNER_SYS_PROMPT,
)

#### 2. Venue planner

In [25]:
VENUE_PLANNER_SYS_PROMPT = """
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    """

venue_agent = create_agent(
    model=OLLAMA_MODEL,
    tools=[web_search],
    system_prompt=VENUE_PLANNER_SYS_PROMPT,
)

#### 3. DJ

In [39]:
DJ_SYS_PROMPT = """
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.
    You may need to make multiple queries to iteratively find the best options.
    NOTE: Make sure to enlist the song choices, alongwith all of the details associated with the input genre from the customer.
    """

playlist_agent = create_agent(
    model=OLLAMA_MODEL,
    tools=[query_playlist_db],
    system_prompt=DJ_SYS_PROMPT,
)

### Orchestrator agent
- The `coordinator` in charge of the **three** subagents.


In [40]:
@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke(
        {
            "messages": [
                HumanMessage(content=f"Find flights from {origin} to {destination}.")
            ]
        }
    )
    return response["messages"][-1].content


@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in the {destination} that can accomodate {capacity} guests."
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content


@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist."
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content


@tool
def update_state(
    origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime
) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre"""
    return Command(
        update={
            "origin": origin,
            "destination": destination,
            "guest_count": guest_count,
            "genre": genre,
            "messages": [
                ToolMessage(
                    "Successfully updated state!", tool_call_id=runtime.tool_call_id
                )
            ],
        }
    )  # type: ignore

In [41]:
COORD_SYS_PROMPT = """
    You are a wedding coordinator. Delegate tasks to your specialists for flights, venues and playlists.
    First find all the information you need to update the state. Once that is done you can delegate the tasks.
    Once you have received their answers, coordinate the perfect wedding for me.
    """

coordinator = create_agent(
    model=OLLAMA_MODEL,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt=COORD_SYS_PROMPT,
)

## Test

In [42]:
response = await coordinator.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="I am from New York and I would like a wedding in Paris for 120 guests. I am into Pop music."
            )
        ],
    }
)

In [43]:
Markdown(response["messages"][-1].content)

Everything is set for your dream wedding in Paris! I have coordinated with my specialists to ensure every detail is covered. Here is your complete wedding plan:

### ✈️ Travel Arrangements
To get your guests from **New York to Paris**, I recommend the **Air France flight from Newark (EWR)**. 
*   **Details:** Departs June 15, 2027, at 17:00 and arrives at 06:10 the next morning.
*   **Why:** It is the shortest travel duration (7h 10m) and offers the reliability of a premium carrier for a very competitive price (~391 EUR).

### 🏰 The Venue
For a guest list of **120 people**, I have selected the best options depending on the vibe you want:
*   **The Classic Estate Choice:** **Le Château Charmant**. It is an exact match for your capacity and offers a stunning French estate feel with a barn and courtyard.
*   **The Luxury Choice:** If you prefer to stay in the heart of the city with iconic views, the **Shangri-La Paris** is the gold standard.
*   **The Logistic Choice:** **Château de Serans** is ideal if you want your guests to stay overnight on-site.

### 🎵 The Soundtrack
Since you love **Pop music**, I've curated a playlist that blends romantic sentiment with upbeat energy. 
*   **Highlights:** The list includes timeless tracks like *"Imagine"* and *"Real Love"* for the ceremony, and higher-energy pop selections to keep the dance floor moving.
*   **Total Duration:** ~37 minutes of curated hits to kick off your celebration.

**Coordinator's Final Touch:** 
Since you're hosting 120 guests in June, the weather will be breathtaking. I suggest **Le Château Charmant** for a romantic, "fairytale" French countryside experience that will leave your New York guests spellbound.

Would you like me to proceed with any bookings or refine the playlist further?

### Probing query
- Validating the o/p

In [ ]:
db_uri = rf"sqlite:///resources/Chinook.db"
engine = create_engine(db_uri)

probe_query = """select t.Name, g.Name
from Track as t
right join Genre g
on t.GenreId = g.GenreId
where t.Name = 'Real Love' and g.Name = 'Pop';"""

with engine.connect() as connection:
    result = connection.execute(text(probe_query))
    for row in result:
        print(row)

('Real Love', 'Pop')
